# Dog Survey Data Cleaning Pipeline (Strict Main.py Replica)

## IMPORTANT PURPOSE OF THIS NOTEBOOK
This notebook is a **direct, corrected replication of `main.py` logic**.

It fixes real issues observed in the previous pipeline:
- ❌ `title` column becoming empty
- ❌ negative values still appearing in `amount_spent_on_dog_food`

## ROOT CAUSE OF BUGS (EXPLAINED SIMPLY)
1. **Title issue**: values were not consistently normalized before validation → valid titles were rejected.
2. **Amount issue**: negative filtering existed, but dirty formats + coercion order caused invalid values to slip through.

## THIS VERSION FIXES THAT BY:
- enforcing strict normalization FIRST
- standardizing column names EARLY
- applying cleaning in correct dependency order
- ensuring invalid numeric values are always removed BEFORE filling median


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../'))

import pandas as pd
import numpy as np

from src.scripts.profiling import dataset_summary, inconsistency_summary
from src.scripts.completeness import completeness_summary
from src.scripts.consistency import fix_dog_size, fix_dog_gender, fix_email
from src.scripts.duplicates import remove_duplicates


# PHASE 1 — DATA LOADING & HARD NORMALIZATION

## WHAT HAPPENS HERE
- Load raw CSV
- Standardise column names (lowercase, trimmed)
- Convert ALL known missing-value formats into real NaN
- Remove noise columns (e.g., Unnamed indexes)

## WHY THIS IS IMPORTANT
If this step is wrong, EVERYTHING downstream breaks:
- string cleaning fails
- numeric conversion fails
- duplicate detection becomes unreliable


In [2]:
def normalize_missing(df):
    df = df.copy()

    df = df.replace(
        regex=[
            r'^\s*$',
            r'^-$',
            r'^none$', r'^None$', r'^NONE$',
            r'^na$', r'^NA$', r'^n/a$', r'^N/A$',
            r'^nan$', r'^NaN$', r'^NULL$', r'^null$'
        ],
        value=np.nan
    )
    return df

def drop_noise_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    df = df.loc[:, ~df.columns.str.contains('^unnamed')]
    return df

df = pd.read_csv('../../data/raw/dog_survey.csv')
df = drop_noise_columns(df)
df = normalize_missing(df)

df_raw = df.copy()
df.head()


,id,title,first_name,last_name,email,amount_spent_on_dog_food,dog_size,dog_gender,dog_age
0,1,Mrs,Frasquito,Dene,fdene0@dell.com,£80.05,S,M,53
1,2,Ms,Lawrence,Kardos,lkardos1@diigo.com,£66.30,XL,M,2
2,3,Rev,Lanna,Wintersgill,lwintersgill2@domainmarket.com,£25.84,L,F,71
3,4,Rev,Richmound,Kimmitt,rkimmitt3@jiathis.com,£50.76,M,M,9
4,5,Rev,Valeda,Dallender,vdallender4@theglobeandmail.com,£83.41,XL,F,23


# PHASE 2 — BASELINE DATA PROFILING

## WHAT THIS DOES
- Shows dataset structure BEFORE any cleaning
- Detects inconsistencies
- Provides baseline metrics

## WHY IT MATTERS
This is your **proof of dirty data** before transformation.


In [3]:
print(dataset_summary(df_raw, 'RAW DATA'))
print(inconsistency_summary(df_raw, df_raw))


    Dataset  Rows  Columns  Missing Values  Duplicates
0  RAW DATA   308        9              63           8
        Issue  Before  After  Fixed
0    dog_size     308    308      0
1  dog_gender     302    302      0
2     dog_age       1      1      0
3      amount      15     15      0
4       email       4      4      0


# PHASE 3 — DATA QUALITY AUDIT

## CHECKS PERFORMED
1. Missing values (completeness)
2. Duplicate rows
3. Structural inconsistencies

## WHY THIS EXISTS
You cannot clean blindly. You must know:
- what is missing
- what is duplicated
- what is invalid


In [4]:
print('--- COMPLETENESS ---')
print(completeness_summary(df_raw))

print('\n--- DUPLICATES ---')
dup_raw = df_raw[df_raw.duplicated(keep=False)]
print(dup_raw)


--- COMPLETENESS ---
                          Missing Count  Missing %
title                                30       9.74
amount_spent_on_dog_food             15       4.87
dog_gender                            8       2.60
dog_size                              5       1.62
email                                 4       1.30
dog_age                               1       0.32
id                                    0       0.00
last_name                             0       0.00
first_name                            0       0.00

--- DUPLICATES ---
      id title first_name  last_name                        email  \
179  180   Mrs   Laughton  Kingsmill  lkingsmill4z@slideshare.net   
180  180   Mrs   Laughton  Kingsmill  lkingsmill4z@slideshare.net   
222  222    Mr    Hillary     Fisbey    hfisbey65@marketwatch.com   
223  222    Mr    Hillary     Fisbey    hfisbey65@marketwatch.com   
224  222    Mr    Hillary     Fisbey    hfisbey65@marketwatch.com   
225  222    Mr    Hillary     Fisbe

# PHASE 4 — CLEANING PIPELINE (STRICT ORDER FROM main.py)

## CRITICAL RULE
The order here is NOT optional.

If you change order:
- email validation breaks
- numeric cleaning breaks
- duplicate logic becomes inconsistent

---

## ORDER OF EXECUTION
1. title cleaning (if exists)
2. dog_size fix
3. dog_gender fix
4. email fix
5. amount cleaning
6. dog age cleaning
7. duplicate removal


In [5]:
df_before = df.copy()

# -----------------------------
# TITLE CLEANING (CRITICAL FIX)
# -----------------------------
def clean_title(df):
    df = df.copy()

    if 'title' not in df.columns:
        return df

    s = df['title'].astype('string')

    # normalize garbage values
    s = s.replace([
        '', ' ', '-', 'none', 'null', 'nan',
        'None', 'NULL', 'NaN'
    ], pd.NA)

    s = s.str.strip()

    valid_titles = {'Mr', 'Mrs', 'Ms', 'Dr', 'Rev', 'Honorable'}
    s = s.where(s.isin(valid_titles), pd.NA)

    # FIX: ensure no silent empty collapse
    df['title'] = s.fillna('Unknown')

    return df

# -----------------------------
# STEP 1 — TITLE
# -----------------------------
before = df.copy()
df = clean_title(df)

# -----------------------------
# STEP 2 — DOG SIZE
# -----------------------------
df = fix_dog_size(df)

# -----------------------------
# STEP 3 — DOG GENDER
# -----------------------------
df = fix_dog_gender(df)

# -----------------------------
# STEP 4 — EMAIL
# -----------------------------
df = fix_email(df)

# -----------------------------
# STEP 5 — AMOUNT CLEANING (FIXED LOGIC)
# -----------------------------
df['amount_spent_on_dog_food'] = (
    df['amount_spent_on_dog_food']
    .astype(str)
    .str.replace('£', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df['amount_spent_on_dog_food'] = pd.to_numeric(
    df['amount_spent_on_dog_food'],
    errors='coerce'
)

# CRITICAL FIX: remove negative values BEFORE imputation
df.loc[
    df['amount_spent_on_dog_food'] <= 0,
    'amount_spent_on_dog_food'
] = np.nan

df['amount_spent_on_dog_food'] = df['amount_spent_on_dog_food'].fillna(
    df['amount_spent_on_dog_food'].median()
)

# -----------------------------
# STEP 6 — DOG AGE
# -----------------------------
df['dog_age'] = df['dog_age'].astype(str).str.extract(r'(\d+)', expand=False)
df['dog_age'] = pd.to_numeric(df['dog_age'], errors='coerce')
df.loc[df['dog_age'] <= 0, 'dog_age'] = np.nan
df['dog_age'] = df['dog_age'].fillna(df['dog_age'].median())

# -----------------------------
# STEP 7 — DUPLICATES
# -----------------------------
df, removed_dup = remove_duplicates(df)


# PHASE 5 — POST-CLEANING VALIDATION

## WHAT YOU VERIFY HERE
- Did cleaning actually reduce missing values?
- Are duplicates removed?
- Did inconsistencies drop?

If this fails, your pipeline is mathematically wrong.

In [6]:
print(dataset_summary(df_raw, 'BEFORE'))
print(dataset_summary(df, 'AFTER'))

print('\n--- COMPLETENESS AFTER ---')
print(completeness_summary(df))

print('\n--- INCONSISTENCIES AFTER ---')
print(inconsistency_summary(df_raw, df))

print('\nRows before:', len(df_raw))
print('Rows after:', len(df))
print('Duplicates removed:', len(removed_dup))


  Dataset  Rows  Columns  Missing Values  Duplicates
0  BEFORE   308        9              63           8
  Dataset  Rows  Columns  Missing Values  Duplicates
0   AFTER   300        9               0           0

--- COMPLETENESS AFTER ---
                          Missing Count  Missing %
id                                    0        0.0
title                                 0        0.0
first_name                            0        0.0
last_name                             0        0.0
email                                 0        0.0
amount_spent_on_dog_food              0        0.0
dog_size                              0        0.0
dog_gender                            0        0.0
dog_age                               0        0.0

--- INCONSISTENCIES AFTER ---
        Issue  Before  After  Fixed
0    dog_size     308      0    308
1  dog_gender     302      0    302
2     dog_age       1      0      1
3      amount      15      0     15
4       email       4      0      4

Ro

# PHASE 6 — EXPORT (FINAL STEP)

## WARNING
If your exported file still contains:
- negative amounts
- empty titles

Then your pipeline is NOT correctly executed.

This version guarantees:
- no negative amounts
- no invalid titles (all mapped to Unknown)


In [7]:
output_path = '../../data/processed/dog_survey_cleaned.csv'
df.to_csv(output_path, index=False)

print('Saved cleaned dataset to:', output_path)


Saved cleaned dataset to: ../../data/processed/dog_survey_cleaned.csv
